#  **ClickHouse & PostgreSQL Setup on EC2 with Python Integration**



## **1.  ClickHouse Installation on EC2**

### **Step 1: Update & Install Dependencies**


In [ ]:
sudo apt update
sudo apt install apt-transport-https ca-certificates dirmngr curl software-properties-common -y

### **Step 2: Add ClickHouse APT Repository**

In [ ]:
sudo mkdir -p /etc/apt/keyrings
curl -fsSL https://packages.clickhouse.com/deb/public.key | sudo gpg --dearmor -o /etc/apt/keyrings/clickhouse.gpg

echo "deb [signed-by=/etc/apt/keyrings/clickhouse.gpg] https://packages.clickhouse.com/deb stable main" | sudo tee /etc/apt/sources.list.d/clickhouse.list

### **Step 3: Install ClickHouse Server and Client**

In [ ]:
sudo apt update
sudo apt install clickhouse-server clickhouse-client -y

### **Step 4: Start and Enable ClickHouse on Boot**

In [ ]:
sudo service clickhouse-server start
sudo systemctl enable clickhouse-server

## **2.  Python Setup with ClickHouse**

### **Step 1: Create Virtual Environment**

In [ ]:
python3 -m venv clickhouse_env
source clickhouse_env/bin/activate

### **Step 2: Install ClickHouse Client Library**

In [ ]:
pip install clickhouse-connect

### **Step 3: Python sample code to Insert and Query Data**

Create a file `clickhouse_test.py`:

In [ ]:
import clickhouse_connect

client = clickhouse_connect.get_client(host='localhost', port=8123)

client.command("CREATE DATABASE IF NOT EXISTS AdityaDB")

client.command("""
    CREATE TABLE IF NOT EXISTS AdityaDB.employees (
        id UInt32,
        name String,
        salary Float32
    ) ENGINE = MergeTree()
    ORDER BY id
""")

client.command("""
    INSERT INTO AdityaDB.employees (id, name, salary) VALUES
    (1, 'Alice', 60000.0),
    (2, 'Bob', 55000.5),
    (3, 'Charlie', 72000.25)
""")

rows = client.query('SELECT * FROM AdityaDB.employees').result_rows
for row in rows:
    print(row)

## You’re now inside the interactive ClickHouse shell!

## Quick Test

### Create Table

In [ ]:
CREATE TABLE hello_world (
    id UInt32,
    message String
) ENGINE = MergeTree()
ORDER BY id;

###  Insert Data

In [ ]:
INSERT INTO hello_world VALUES
(1, 'Hello'),
(2, 'ClickHouse'),
(3, 'Rocks!');

### Query Table

In [ ]:
SELECT * FROM hello_world;

Expected output:

```text
┌─id─┬─message──────┐
│  1 │ Hello        │
│  2 │ ClickHouse   │
│  3 │ Rocks!       │
└────┴──────────────┘
```

## **3.  PostgreSQL Installation and Setup on EC2**

### **Step 1: Install PostgreSQL**

In [ ]:
sudo apt update
sudo apt install postgresql postgresql-contrib -y

### **Step 2: Start and Enable PostgreSQL**

In [ ]:
sudo systemctl start postgresql
sudo systemctl enable postgresql

### **Step 3: Switch to Postgres User**

In [ ]:
sudo -i -u postgres

### **Step 4: Create Database and Role (if needed)**

In [ ]:
psql 
CREATE DATABASE TestDB;
\q

## **4.  Transferring Backup from Local to EC2**

### **Step 1: Backup Locally**

In [ ]:
pg_dump -U postgres -F p -f backup.sql TestDB

### **Step 2: Transfer to EC2**

In [ ]:
scp -i "/path/to/Ubuntu.pem" backup.sql ubuntu@<EC2_PUBLIC_IP>:/home/ubuntu/

## **5.  Restore in PostgreSQL on EC2**

### **Step 1: Set File Permissions (if needed)**

In [ ]:
sudo chown postgres:postgres /home/ubuntu/backup.sql

### **Step 2: Switch to Postgres User**

In [ ]:
sudo -i -u  postgres

### **Step 3: Restore Using psql (for plain text format)**

In [ ]:
psql -d testdb -f /home/ubuntu/backup.sql

## 6.  Troubleshooting Tips

- Ensure ClickHouse/PgSQL ports are open in EC2 Security Group (8123 for ClickHouse, 5432 for PostgreSQL).
- Always match `pg_dump` and `psql` versions or use compatible format (`-F p`).
- Use `psql -U postgres -d <db>` instead of peer login when needed.

### PostgreSQL CLI Tips:

- List databases:
  ```bash
  \l
  ```

- List roles:
  ```bash
  \du
  ```